# 04. Volatility Analysis

This notebook covers:
1. Computing volatility indicators
2. Computing renewable share
3. Analyzing relationship between renewables and volatility

Produces:
- **Graph 7**: Volatility vs Renewable Share scatter plot

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src import config
from src import io as data_io
from src import features
from src import models
from src import plots

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Data

In [ ]:
df = data_io.load_processed_data('nsw_processed')
df = features.compute_time_features(df)

print(f"Data: {len(df)} intervals")
print(f"Columns: {list(df.columns)}")

## 2. Compute Volatility Indicators

Three volatility metrics:
1. **1-hour rolling std**: Short-term volatility
2. **6-hour rolling std**: Medium-term volatility
3. **Daily max/mean ratio**: Intra-day volatility

In [ ]:
# Compute rolling volatility
df = features.compute_rolling_volatility(df)

print("Volatility columns added:")
for col in df.columns:
    if 'volatility' in col:
        print(f"  {col}: mean={df[col].mean():.2f}, median={df[col].median():.2f}")

In [ ]:
# Compute daily volatility ratio
daily_vol = features.compute_daily_volatility_ratio(df)

print("\nDaily volatility statistics:")
display(daily_vol.describe())

In [ ]:
# Plot volatility time series
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Price
ax1 = axes[0]
ax1.plot(df.index, df['rrp'], linewidth=0.5, alpha=0.8)
ax1.set_ylabel('RRP ($/MWh)')
ax1.set_title('Price and Volatility Over Time')

# 1h volatility
ax2 = axes[1]
ax2.plot(df.index, df['volatility_1h'], linewidth=0.5, color='orange')
ax2.set_ylabel('1h Rolling Std')

# 6h volatility
ax3 = axes[2]
ax3.plot(df.index, df['volatility_6h'], linewidth=0.5, color='green')
ax3.set_ylabel('6h Rolling Std')
ax3.set_xlabel('Date')

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'volatility_timeseries.png', dpi=150)
plt.show()

## 3. Compute Renewable Share

In [ ]:
# Check if renewable data exists
has_solar = 'solar' in df.columns
has_wind = 'wind' in df.columns

print(f"Solar data available: {has_solar}")
print(f"Wind data available: {has_wind}")

if has_solar or has_wind:
    df = features.compute_renewable_share(df)
    df = features.bin_renewable_share(df)
    
    print(f"\nRenewable share statistics:")
    print(f"  Mean: {df['renewable_share'].mean()*100:.1f}%")
    print(f"  Median: {df['renewable_share'].median()*100:.1f}%")
    print(f"  Max: {df['renewable_share'].max()*100:.1f}%")
    
    # Distribution of renewable share bins
    print(f"\nDistribution by bin:")
    print(df['renewable_share_bin'].value_counts())
else:
    print("\nNo renewable generation data available.")
    print("Skipping renewable-volatility analysis.")
    print("To enable this analysis, add solar/wind generation data.")

In [ ]:
# Plot renewable share over time (if available)
if 'renewable_share' in df.columns:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df.index, df['renewable_share']*100, linewidth=0.5, alpha=0.8)
    ax.set_xlabel('Date')
    ax.set_ylabel('Renewable Share (%)')
    ax.set_title('Renewable Energy Share Over Time')
    plt.tight_layout()
    plt.show()

## 4. Correlation Analysis

In [ ]:
if 'renewable_share' in df.columns:
    # Compute correlations
    volatility_cols = ['volatility_1h', 'volatility_6h']
    
    corr_results = models.compute_volatility_renewable_correlation(
        df, volatility_cols, 'renewable_share'
    )
    
    print("Correlation between volatility and renewable share:")
    display(corr_results)
else:
    print("Renewable share not available for correlation analysis.")

In [ ]:
# Compare volatility by renewable share bin
if 'renewable_share_bin' in df.columns:
    for vol_col in ['volatility_1h', 'volatility_6h']:
        binned = models.compare_volatility_by_renewable_bin(df, vol_col)
        print(f"\n{vol_col} by renewable share bin:")
        display(binned)

## 5. Graph 7: Volatility vs Renewable Share

In [ ]:
if 'renewable_share' in df.columns:
    fig = plots.plot_volatility_vs_renewable(
        df,
        volatility_col='volatility_1h',
        renewable_col='renewable_share'
    )
    plt.show()
else:
    print("Cannot create volatility vs renewable plot without renewable data.")

## 6. Daily Volatility Analysis

In [ ]:
# Plot daily volatility ratio over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Volatility ratio
ax1 = axes[0]
ax1.plot(daily_vol.index, daily_vol['volatility_ratio'], 'o-', markersize=2, alpha=0.7)
ax1.axhline(y=daily_vol['volatility_ratio'].median(), color='red', linestyle='--', label='Median')
ax1.set_ylabel('Max/Mean Price Ratio')
ax1.set_title('Daily Intra-day Volatility')
ax1.legend()

# Price range
ax2 = axes[1]
ax2.plot(daily_vol.index, daily_vol['price_range'], 'o-', markersize=2, alpha=0.7, color='orange')
ax2.set_xlabel('Date')
ax2.set_ylabel('Price Range ($/MWh)')
ax2.set_title('Daily Price Range (Max - Min)')

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'daily_volatility.png', dpi=150)
plt.show()

In [ ]:
# Most volatile days
print("Top 10 most volatile days (by max/mean ratio):")
display(daily_vol.nlargest(10, 'volatility_ratio'))

## 7. Volatility by Time of Day

In [ ]:
# Average volatility by hour
hourly_vol = df.groupby('hour')[['volatility_1h', 'volatility_6h']].mean()

fig, ax = plt.subplots(figsize=(12, 5))
hourly_vol.plot(ax=ax, marker='o')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Mean Volatility')
ax.set_title('Average Volatility by Hour')
ax.set_xticks(range(24))
ax.legend(['1h Rolling Std', '6h Rolling Std'])
plt.tight_layout()
plt.show()

## 8. Save Enhanced Data

In [ ]:
# Save data with volatility features
data_io.save_processed_data(df, 'nsw_with_volatility')
print(f"Saved enhanced data with volatility features.")

## 9. Key Findings

Record your findings:

- **Is there a correlation between renewable share and volatility?**
- **Direction of relationship:**
- **Which hours are most volatile?**
- **Seasonal volatility patterns:**

---
## Next Steps
Proceed to `05_case_studies.ipynb` for detailed spike event case studies.